In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [34]:
from nbqol import path_to_git_root

dir = path_to_git_root() # automatically makes path to repo directory (wherever you cloned it)

print("Repository directory: ", dir)

# type(dir)

Repository directory:  /mnt/c/Users/Timothy/Documents/Chou


Pairwise comparisons
1. Load each of 3 datasets
2. Do pairwise comparisons for genes:
    - upregulated r vs v; upregulated r vs u; upregulated r vs u; upregulated u vs v
    - downregulated r vs v; downregulated r vs u; downregulated r vs u; downregulated u vs v
    - take ~top 25 genes
3. check for overlap of genes b/w each comparison

In [37]:
rp_rr_df = pd.read_csv(dir+'/data/rp_vs_rr/rp_vs_rr_summary.txt', sep='\t')
up_ur_df = pd.read_csv(dir+'/data/up_vs_ur/up_vs_ur_summary.txt', sep='\t')
vp_vr_df = pd.read_csv(dir+'/data/vp_vs_vr/vp_vs_vr_summary.txt', sep='\t')

rp_rr_df['line'] = 'RT112'
up_ur_df['line'] = 'UMUC1'
vp_vr_df['line'] = '647V'

# Combine into single df and save as csv
combined_df = pd.concat([rp_rr_df, up_ur_df, vp_vr_df])
combined_df.to_csv(dir+'/data/combined-lines_summary.txt', index=False)
del combined_df # just freeing up memory

#drop missing padj, log2FC, Genes
rp_rr_df = rp_rr_df.dropna(subset=["padj", "log2FoldChange", "GeneSymbol"])
up_ur_df = up_ur_df.dropna(subset=["padj", "log2FoldChange", "GeneSymbol"])
vp_vr_df = vp_vr_df.dropna(subset=["padj", "log2FoldChange", "GeneSymbol"])

In [18]:
print("RT112:", len(rp_rr_df), "genes")
print("UMUC1:", len(up_ur_df), "genes")
print("V:", len(vp_vr_df), "genes")

RT112: 24045 genes
UMUC1: 23364 genes
V: 12542 genes


In [ ]:
# Combine into single df



#### Get T25 upregulated and downregulated genes per dataset

In [19]:
# Filter to only log2FC >0 and sort by top 25 significant genes
r_up = rp_rr_df[rp_rr_df['log2FoldChange'] > 1].sort_values('padj').head(100)
u_up = up_ur_df[up_ur_df['log2FoldChange'] > 1].sort_values('padj').head(100)
v_up = vp_vr_df[vp_vr_df['log2FoldChange'] > 1].sort_values('padj').head(100)

r_down = rp_rr_df[rp_rr_df['log2FoldChange'] < -1].sort_values('padj').head(100)
u_down = up_ur_df[up_ur_df['log2FoldChange'] < -1].sort_values('padj').head(100)
v_down = vp_vr_df[vp_vr_df['log2FoldChange'] < -1].sort_values('padj').head(100)

Pairwise overlaps

In [20]:
r_up_genes   = set(r_up["GeneSymbol"])
r_down_genes = set(r_down["GeneSymbol"])
u_up_genes   = set(u_up["GeneSymbol"])
u_down_genes = set(u_down["GeneSymbol"])
v_up_genes   = set(v_up["GeneSymbol"])
v_down_genes = set(v_down["GeneSymbol"])

In [21]:
#upregulated overlaps
r_u_upregulated_overlap = r_up_genes & u_up_genes
r_v_upregulated_overlap = r_up_genes & v_up_genes
u_v_upregulated_overlap = u_up_genes & v_up_genes

#downregulated overlaps
r_u_downregulated_overlap = r_down_genes & u_down_genes
r_v_downregulated_overlap = r_down_genes & v_down_genes
u_v_downregulated_overlap = u_down_genes & v_down_genes

In [22]:
print(' # overlapping upregulated genes:')
print('UMUC1 + 647V:', len(u_v_upregulated_overlap))
print('RT112 + UMUC1:', len(r_u_upregulated_overlap))
print('RT112 + 647V:', len(r_v_upregulated_overlap))

print('# overlapping downregulated genes:')
print('UMUC1 + 647V:', len(u_v_downregulated_overlap))
print('RT112 + UMUC1:', len(r_u_downregulated_overlap))
print('RT112 + 647V:', len(r_v_downregulated_overlap))

 # overlapping upregulated genes:
UMUC1 + 647V: 1
RT112 + UMUC1: 0
RT112 + 647V: 1
# overlapping downregulated genes:
UMUC1 + 647V: 1
RT112 + UMUC1: 5
RT112 + 647V: 4


In [23]:
print('Upregulated genes in UMUC1 + 647V:', u_v_upregulated_overlap)
print('Upregulated genes in RT112 + UMUC1:', r_u_upregulated_overlap)
print('Upregulated genes in RT112 + 647V:', r_v_upregulated_overlap)

print('Downregulated genes in UMUC1 + 647V:', u_v_downregulated_overlap)
print('Downregulated genes in RT112 + UMUC1:', r_u_downregulated_overlap)
print('Downregulated genes in RT112 + 647V:', r_v_downregulated_overlap)


Upregulated genes in UMUC1 + 647V: {'EEF1A2'}
Upregulated genes in RT112 + UMUC1: set()
Upregulated genes in RT112 + 647V: {'COL4A6'}
Downregulated genes in UMUC1 + 647V: {'UPK1B'}
Downregulated genes in RT112 + UMUC1: {'GLUL', 'MMRN2', 'HEY1', 'RHOU', 'IGFBP3'}
Downregulated genes in RT112 + 647V: {'STC2', 'WNT10A', 'TGFBI', 'H19'}


#### Visualize

In [24]:
rp_rr_df

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,GeneSymbol,gene_type
0,1933.153304,-3.375006,0.083720,-40.312927,0.000000,0.000000,F3,protein_coding
1,7291.676853,5.953054,0.084600,70.367366,0.000000,0.000000,TXNIP,protein_coding
2,1894.517793,-4.164389,0.099914,-41.679786,0.000000,0.000000,DHRS9,protein_coding
3,1409.878109,-4.876914,0.118682,-41.092419,0.000000,0.000000,AREG,protein_coding
4,1662.933358,-3.886193,0.092667,-41.937209,0.000000,0.000000,SLC7A11,protein_coding
...,...,...,...,...,...,...,...,...
24040,673.297854,0.000052,0.101705,0.000514,0.999590,0.999756,NEURL4,protein_coding
24041,2.990742,-0.000347,1.231861,-0.000281,0.999775,0.999900,RP4-612C19.2,processed_pseudogene
24042,4.795836,-0.000130,1.060534,-0.000122,0.999902,0.999902,RP11-392O17.2,lincRNA
24043,22.054377,-0.000058,0.462696,-0.000125,0.999900,0.999902,LINC00216,lincRNA
